# 02 – Data exploration

This notebook presents the eight exploration questions. For each question it
gives the definitions used, the method, why this method was chosen (and the
alternatives that were considered), and the results.

**Prerequisites.** The results are computed by `jobs/exploration.py` (about
15 minutes for all questions) and written to `data/results/`. Q6 also needs
the original languages downloaded from Wikidata:

```bash
docker compose exec spark python jobs/exploration.py q6        # writes the Q6 targets
docker compose exec spark python scripts/fetch_languages.py    # about 1 minute
docker compose exec spark python jobs/exploration.py           # all questions
```

The code of every question is in `src/imdb_project/exploration.py`. All
questions read the clean (silver) tables built by the data preparation step
(see `01_data_preparation.ipynb`).

In [1]:
import json

import pandas as pd
from pyspark.sql import functions as F

from imdb_project import exploration as ex
from imdb_project.config import ENRICHMENT_DIR, RESULTS_DIR
from imdb_project.io import read_silver
from imdb_project.spark import get_spark

spark = get_spark("notebook-exploration")
spark.sparkContext.setLogLevel("ERROR")
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 60)


def result(name):
    """Read a result table written by jobs/exploration.py."""
    path = RESULTS_DIR / name / "parquet"
    if not path.exists():
        raise FileNotFoundError(f"Run jobs/exploration.py first ({name} is missing)")
    return spark.read.parquet(str(path))


titles = read_silver(spark, "title.basics")
ratings = read_silver(spark, "title.ratings")

## Common definitions

Several questions share the same definitions. They are constants at the top
of `src/imdb_project/exploration.py`, so they can be changed in one place.

| Definition | Value | Why |
|---|---|---|
| Feature film | `titleType = movie` and runtime over `FEATURE_MIN_RUNTIME` = 40 minutes | Definition used by the Academy of Motion Picture Arts and Sciences, the American Film Institute and the British Film Institute |
| Reliable rating | at least `MIN_VOTES` = 1,000 votes | A rating given by a handful of people (often the cast and crew's relatives) is not comparable with a rating given by thousands |
| Successful film (Q6) | rating >= `SUCCESS_MIN_RATING` = 8.0 with >= `SUCCESS_MIN_VOTES` = 1,000 votes | Well rated, by enough people |

The 1,000-vote threshold is strict: it keeps about 15 % of the rated feature
films. Older films and films from smaller industries are less represented.

In [2]:
from imdb_project.definitions import FEATURE_MIN_RUNTIME

print("FEATURE_MIN_RUNTIME =", FEATURE_MIN_RUNTIME)
print("MIN_VOTES =", ex.MIN_VOTES)
print("SUCCESS_MIN_RATING =", ex.SUCCESS_MIN_RATING, "| SUCCESS_MIN_VOTES =", ex.SUCCESS_MIN_VOTES)

FEATURE_MIN_RUNTIME = 40
MIN_VOTES = 1000
SUCCESS_MIN_RATING = 8.0 | SUCCESS_MIN_VOTES = 1000


## Q1 – Feature films whose runtime is far from their genre's

> Find all feature-length movies by genre whose length is notably different
> than the average length of movies in that genre.

**Method: Tukey fences (interquartile range).** For each genre, `Q1` and
`Q3` are the first and third quartiles of the runtimes and `IQR = Q3 - Q1`.
A film is an outlier in a genre when its runtime is outside
`[Q1 - 3 * IQR, Q3 + 3 * IQR]` (`k = 3` marks "far out" values). A film
with several genres is tested in each of them; genres with fewer than 30
films are skipped.

**Why this method.** The first idea was a z-score: a film is an outlier
when it is more than 3 standard deviations away from the genre's mean. But
the mean and the standard deviation are themselves pulled by the extreme
values: in the Documentary genre, a single 857-hour film (*Logistics*)
raised the standard deviation to 223 minutes, so that only 31 documentaries
were flagged. Quartiles do not move when a few extreme values are added, so
the films can be compared with their genre without removing anything first.

**Alternatives considered.**

- z-score on the runtime (pulled by the extreme values, see above);
- z-score on the logarithm of the runtime (less sensitive, but harder to
  read);
- IQR on the logarithm of the runtime (finds a few "shorter" films, bounds
  harder to read);
- formal outlier tests such as Grubbs' test: they assume normally
  distributed values (runtimes are not, they are skewed towards long
  values) and are designed to find one or two outliers in a small sample;
- modified z-score based on the median absolute deviation (Iglewicz and
  Hoaglin), another robust method giving similar results.

The IQR on the runtime in minutes was kept because its bounds can be read
directly in minutes.

**Films over 24 hours.** The 209 titles flagged `runtime_over_24h` during
the preparation are kept: the quartiles, and therefore the bounds, are the
same with and without them (checked below).

In [3]:
stats = result("q1_genre_runtime_stats").orderBy(F.desc("films")).toPandas()
stats

,genre,films,avg_runtime,median_runtime,q1_min,q3_min,lower_bound_min,upper_bound_min,longer,shorter,outliers
0,Drama,196714,96.1,93.0,81.0,107.0,3.0,185.0,956,0,956
1,Documentary,116146,78.0,74.0,57.0,90.0,-42.0,189.0,748,0,748
2,Comedy,96649,93.1,91.0,82.0,102.0,22.0,162.0,716,0,716
3,Romance,42442,99.1,95.0,85.0,110.0,10.0,185.0,128,0,128
4,Action,40375,101.2,94.0,85.0,113.0,1.0,197.0,309,0,309
5,Crime,34643,95.5,92.0,83.0,105.0,17.0,171.0,284,0,284
6,Thriller,34096,96.1,93.0,85.0,105.0,25.0,165.0,209,0,209
7,Horror,30748,88.6,88.0,79.0,96.0,28.0,147.0,326,0,326
8,Adventure,24063,93.0,90.0,79.0,101.0,13.0,167.0,436,0,436
9,Mystery,16483,94.3,92.0,81.0,105.0,9.0,177.0,88,0,88


In [4]:
# The bounds are the same when the films over 24 hours are left out.
_, stats_without, excluded = ex.runtime_outliers_by_genre(
    titles, excluded_flags=["runtime_over_24h"])
compare = stats[["genre", "lower_bound_min", "upper_bound_min"]].merge(
    stats_without.select("genre", "lower_bound_min", "upper_bound_min").toPandas(),
    on="genre", suffixes=("", "_without_24h"))
print("Genres whose bounds change:",
      int(((compare["lower_bound_min"] != compare["lower_bound_min_without_24h"])
           | (compare["upper_bound_min"] != compare["upper_bound_min_without_24h"])).sum()))

Genres whose bounds change: 0


In [5]:
outliers = result("q1_runtime_outliers")
print(f"{outliers.count():,} (genre, film) outliers, "
      f"{outliers.select('tconst').distinct().count():,} distinct films")
outliers.groupBy("direction").count().toPandas()

5,278 (genre, film) outliers, 3,730 distinct films


,direction,count
0,longer,5278


In [6]:
outliers.orderBy(F.desc("iqr_distance")).limit(15).toPandas()

,genre,tconst,primaryTitle,startYear,runtimeMinutes,genre_median_runtime,iqr_distance,direction
0,Documentary,tt8273150,Logistics,2012,51420,74.0,1555.45,longer
1,Documentary,tt3854496,Ambiancé,2020,43200,74.0,1306.36,longer
2,Documentary,tt12277054,Carnets Filmés (Liste Complète),2019,28643,74.0,865.24,longer
3,Documentary,tt2659636,Modern Times Forever,2011,14400,74.0,433.64,longer
4,Documentary,tt2355497,Beijing 2003,2004,9000,74.0,270.00,longer
5,Comedy,tt5136218,London EC1,2015,5460,91.0,267.90,longer
6,Drama,tt5068890,Hunger!,2015,6000,93.0,226.65,longer
7,Mystery,tt5136218,London EC1,2015,5460,92.0,223.13,longer
8,Drama,tt1806963,Matrjoschka,2006,5700,93.0,215.12,longer
9,Drama,tt5136218,London EC1,2015,5460,93.0,205.88,longer


**Why are there no "shorter" films?** The lower bound is below 40 minutes
in every genre, so no feature film can be under it. Two reasons:

- the feature film definition removes every film of 40 minutes or less,
  while the median runtime is about 90 minutes;
- runtimes are skewed towards long values: a film can last ten times the
  median, never ten times less.

## Q2 – Top three genres per year (1980-2025)

> What are the top three most popular movie genres for each year between
> 1980 and 2025?

**Method.** Rated feature films released between 1980 and 2025. A film with
several genres counts fully in each of them. "Popularity" has no single
definition, so three measures are compared:

| Measure | What it measures | Bias |
|---|---|---|
| Total votes of the genre's films | audience | favours genres that produce many films |
| Number of films | production | measures supply rather than popularity |
| Plain average rating of the genre's films with >= 1,000 votes (genres with at least 10 such films that year) | appreciation | every film weighs the same; favours niche genres watched by an interested public |

The average rating weighted by the votes (`weighted_rating`) is also in the
table but is not used for a ranking: it is dominated by the same blockbusters
as the vote total.

`numVotes` is the number of votes today, not at the release: recent films,
and films that became classics, are favoured.

In [7]:
popularity = result("q2_genre_popularity_by_year")
tops = {
    m: result(f"q2_top3_by_{m}").toPandas().set_index("year")
    for m in ex.POPULARITY_METHODS
}
side_by_side = pd.concat(
    {m: df[["genre_1", "genre_2", "genre_3"]] for m, df in tops.items()}, axis=1)
side_by_side

votes                         films                             avg_rating                        
        genre_1    genre_2    genre_3 genre_1      genre_2      genre_3      genre_1      genre_2    genre_3
year                                                                                                        
1980      Drama  Adventure     Horror   Drama       Comedy       Action    Biography      Romance      Music
1981  Adventure      Drama     Horror   Drama       Comedy       Action        Music    Animation    History
1982      Drama     Sci-Fi     Action   Drama       Comedy       Action    Animation      History      Drama
1983      Drama     Action  Adventure   Drama       Comedy       Action    Biography        Drama    Mystery
1984     Action      Drama  Adventure   Drama       Comedy       Action  Documentary      History      Drama
1985     Comedy  Adventure      Drama   Drama       Comedy       Action    Biography    Animation      Drama
1986      Drama  Adventure     Comedy   Drama       Comedy       Action    Animation       Family      Drama
1987      Drama     Comedy     Action   Drama       Comedy       Action    Biography          War  Animation
1988      Drama     Comedy     Action   Drama       Comedy       Action  Documentary    Animation  Biography
1989     Comedy      Drama  Adventure   Drama       Comedy       Action      History    Animation        War
1990      Drama     Comedy      Crime   Drama       Comedy       Action    Biography      History        War
1991      Drama      Crime     Comedy   Drama       Comedy       Action    Biography    Animation      Drama
1992      Drama     Comedy      Crime   Drama       Comedy       Action      History        Music  Biography
1993      Drama     Comedy      Crime   Drama       Comedy       Action      History    Biography  Animation
1994      Drama      Crime     Comedy   Drama       Comedy       Action    Biography    Animation      Drama
1995      Drama      Crime     Comedy   Drama       Comedy       Action    Animation      History  Biography
1996      Drama     Comedy      Crime   Drama       Comedy       Action          War        Music  Biography
1997      Drama     Action     Comedy   Drama       Comedy      Romance  Documentary    Animation  Biography
1998      Drama     Comedy      Crime   Drama       Comedy      Romance          War  Documentary    History
1999      Drama     Comedy      Crime   Drama       Comedy      Romance  Documentary    Biography    History
2000      Drama     Comedy  Adventure   Drama       Comedy      Romance  Documentary          War    History
2001      Drama     Comedy  Adventure   Drama       Comedy      Romance  Documentary    Animation    History
2002      Drama  Adventure     Action   Drama       Comedy  Documentary  Documentary    Biography    History
2003      Drama     Action     Comedy   Drama       Comedy  Documentary  Documentary      History  Biography
2004      Drama     Action     Comedy   Drama       Comedy  Documentary    Biography  Documentary      Sport
2005      Drama     Comedy     Action   Drama       Comedy  Documentary  Documentary    Biography        War
2006      Drama     Comedy   Thriller   Drama       Comedy  Documentary  Documentary      History  Biography
2007      Drama     Comedy     Action   Drama       Comedy  Documentary  Documentary    Biography        War
2008      Drama     Action     Comedy   Drama       Comedy  Documentary  Documentary    Biography    History
2009      Drama     Comedy     Action   Drama  Documentary       Comedy  Documentary    Biography    History
2010      Drama     Action  Adventure   Drama  Documentary       Comedy  Documentary    Biography        War
2011      Drama     Action     Comedy   Drama  Documentary       Comedy  Documentary        Sport  Biography
2012      Drama     Action     Comedy   Drama  Documentary       Comedy  Documentary    Biography      Music
2013      Drama     Action     Comedy   Drama  Documentary       Comedy  Documentary  

In [8]:
frequency = {m: result(f"q2_top3_frequency_by_{m}").toPandas() for m in ex.POPULARITY_METHODS}
for method, df in frequency.items():
    print(f"\n--- Years in the top 3, popularity = {method}")
    print(df.to_string(index=False))


--- Years in the top 3, popularity = votes
    genre  years_in_top  years_first  first_year  last_year
    Drama            46           42        1980       2025
   Comedy            34            2        1985       2025
   Action            29            1        1982       2025
Adventure            16            1        1980       2021
    Crime             9            0        1990       1999
   Horror             2            0        1980       1981
 Thriller             1            0        2006       2006
   Sci-Fi             1            0        1982       1982

--- Years in the top 3, popularity = films
      genre  years_in_top  years_first  first_year  last_year
      Drama            46           46        1980       2025
     Comedy            46            0        1980       2025
Documentary            24            0        2002       2025
     Action            17            0        1980       1996
    Romance             5            0        1997       2001


## Q3 – Top three genres compared with the other genres

> How do the top three genres compare to the rest of the genres of movies in
> a given year?

**Method.** For each year, the top three genres by total votes are compared
with all the other genres: share of the votes, share of the films, votes per
film, and average rating (films with >= 1,000 votes). Votes and films are
summed per genre, so a film with several genres is counted once per genre.

In [9]:
top_vs_rest = result("q3_top3_vs_rest").toPandas()
top_vs_rest

,year,top_vote_share_pct,top_film_share_pct,top_votes_per_film,rest_votes_per_film,top_avg_rating,rest_avg_rating,other_genres
0,1980,44.3,37.0,5699.0,4207.0,6.08,6.25,21
1,1981,46.5,35.7,3884.0,2484.0,6.22,6.52,20
2,1982,43.7,38.6,5032.0,4072.0,6.12,6.36,19
3,1983,45.9,41.0,4327.0,3558.0,6.05,6.16,19
4,1984,44.2,39.4,6199.0,5094.0,6.24,6.32,19
5,1985,51.5,46.8,5973.0,4943.0,6.36,6.21,19
6,1986,46.2,47.3,5018.0,5233.0,6.38,6.26,19
7,1987,42.6,52.9,4437.0,6701.0,6.14,6.25,19
8,1988,46.8,51.4,5180.0,6225.0,6.23,6.32,19
9,1989,49.5,45.7,5821.0,4998.0,6.31,6.24,19


In [10]:
print("Years where the top 3 is rated lower than the other genres:",
      int((top_vs_rest["top_avg_rating"] < top_vs_rest["rest_avg_rating"]).sum()),
      "of", len(top_vs_rest))
top_vs_rest[["top_vote_share_pct", "top_film_share_pct"]].astype(float).describe()

Years where the top 3 is rated lower than the other genres: 34 of 46


,top_vote_share_pct,top_film_share_pct
count,46.000000,46.000000
mean,44.754348,43.645652
std,3.479700,6.344681
min,38.300000,30.800000
25%,42.700000,39.800000
50%,44.500000,44.900000
75%,46.275000,49.175000
max,58.300000,52.900000


## Q4 – Directors with a single hit

> List all directors who have directed exactly one movie with a rating >= 8.0
> but who have multiple other movies directed with an average rating of
> <= 6.0.

**Method.** Feature films with at least 1,000 votes; directors come from
`title.crew` (complete list of directors of each film). A director is kept
when they have exactly one film rated >= 8.0 and at least two other films,
**each** rated <= 6.0.

**Why.** Two readings of "multiple other movies with an average rating of
<= 6.0" are possible: the average of the other films is <= 6.0, or every
other film is <= 6.0. The strict reading was chosen: a director whose other
films include a 6.8 is not a "one-hit" director, even if the average is low.
With the average reading, the list had 48 directors (including M. Night
Shyamalan, Irvin Kershner and Robert Rodriguez).

"Multiple" is read as "at least two". IMDb classifies concert films and TV
specials as `movie`, so some appear in the list.

In [11]:
result("q4_one_hit_directors").toPandas()

,nconst,primaryName,hit_title,hit_year,hit_rating,hit_votes,other_movies,other_avg_rating,other_best_rating
0,nm0446819,Richard Kelly,Donnie Darko,2001,8.0,925161,2,5.45,5.6
1,nm0204628,Garth Davis,Lion,2016,8.0,270263,2,5.65,5.9
2,nm0244892,J.P. Dutta,Border,1997,8.0,20775,4,5.33,5.5
3,nm8678191,Karthick Naren,Dhuruvangal Pathinaaru,2016,8.1,19410,2,5.15,5.9
4,nm6044360,Ravikanth Perepu,Kshanam,2016,8.1,6708,2,5.95,6.0
5,nm0002239,Prince,Sign 'o' the Times,1987,8.0,3054,2,4.90,5.1
6,nm0389514,David Hogan,Sign 'o' the Times,1987,8.0,3054,2,4.60,5.6
7,nm1048560,Marcus Raboy,Conan O'Brien: The Kennedy Center Mark Twain Prize for American Humor,2025,8.0,1584,2,5.45,5.8


## Q5 – Series whose worst episode is in the last season

> Identify television series with at least three seasons consisting of >=10
> episodes where the worst episode occurred in the last season and the best
> episode occurred within the first two seasons.

**Method.**

- `tvSeries` and `tvMiniSeries`; season 0 (specials) is ignored and the last
  season is the highest season number;
- at least three seasons of 10 episodes or more (other seasons may be
  shorter);
- only episodes with at least `EPISODE_MIN_VOTES` = 100 votes are compared,
  and the series needs at least `MIN_RATED_EPISODES` = 10 of them;
- ties are strict: **every** episode sharing the lowest rating must be in
  the last season, and every episode sharing the highest rating must be in
  season 1 or 2.

**Why a lower vote threshold than for films.** IMDb has many more episodes
than films, and the votes of a series are spread over all its episodes:
viewers rate the series more often than each episode. The median episode
has about 22 votes, against 78 for a feature film, and only about 4 % of the
rated episodes reach 1,000 votes (see below). Without any threshold, the
result was dominated by obscure series whose episodes were rated 1.0 or 10.0
by a few people.

In [12]:
kind = (F.when((F.col("titleType") == "movie") & (F.col("runtimeMinutes") > 40), "feature film")
        .when(F.col("titleType") == "tvEpisode", "tv episode"))
(titles.select("tconst", kind.alias("kind")).where(F.col("kind").isNotNull())
 .join(ratings, "tconst")
 .groupBy("kind")
 .agg(F.count(F.lit(1)).alias("rated_titles"),
      F.percentile("numVotes", 0.5).alias("median_votes"),
      F.round(F.avg((F.col("numVotes") >= 100).cast("int")) * 100, 1).alias("pct_100_votes_or_more"),
      F.round(F.avg((F.col("numVotes") >= 1000).cast("int")) * 100, 1).alias("pct_1000_votes_or_more"))
 .toPandas())

,kind,rated_titles,median_votes,pct_100_votes_or_more,pct_1000_votes_or_more
0,feature film,321692,78.0,45.5,15.4
1,tv episode,895577,22.0,21.3,4.2


In [13]:
series = result("q5_series_worst_last_best_first")
print(f"{series.count():,} series")
series.orderBy(F.desc("best_averageRating")).limit(25).toPandas()

133 series


,tconst,series_title,series_start,seasons,full_seasons,last_season,rated_episodes,best_seasonNumber,best_episodeNumber,best_episode_title,best_averageRating,best_ties,worst_seasonNumber,worst_episodeNumber,worst_episode_title,worst_averageRating,worst_ties
0,tt9335498,Demon Slayer: Kimetsu no Yaiba,2019,4,3,4,63,2,17,Never Give Up,9.8,1,4,4,To Bring a Smile to One's Face,7.1,1
1,tt12063450,To Your Eternity,2021,3,3,3,62,1,12,Awakening,9.7,1,3,21,Stay Here,6.6,1
2,tt1641384,Young Justice,2010,4,4,4,98,2,19,Summit,9.7,1,4,14,Nautical Twilight,6.8,1
3,tt4839610,We Bare Bears,2014,4,4,4,138,2,16,Yuri and the Bear,9.6,1,4,34,Ranger Norm,6.3,1
4,tt3107288,The Flash,2014,9,9,9,184,1,15,Out of Time,9.6,1,9,6,"The Good, the Bad and the Lucky",3.8,1
5,tt1856010,House of Cards,2013,6,5,6,73,2,1,Chapter 14,9.5,2,6,8,Chapter 73,2.5,1
6,tt4508902,One Punch Man,2015,3,3,3,36,1,12,The Strongest Hero,9.5,1,3,6,Motley Heroes,1.6,1
7,tt0934814,Chuck,2007,5,5,5,91,2,22,Chuck Versus the Ring,9.5,1,5,2,Chuck Versus the Bearded Bandit,7.4,1
8,tt13675832,As If,2021,6,6,6,68,2,5,Çaça ve Cosplay,9.4,1,6,3,Girls Night Out,6.2,1
9,tt8879940,Mythic Quest,2020,4,3,4,40,1,5,A Dark Quiet Death,9.4,2,4,8,Rebrand,6.5,1


## Q6 – Actors successful in several languages

> Find any actors/actresses who have achieved success across films in
> multiple distinct languages. Additionally, list all actors/actresses who
> have achieved success in movies in >=3 languages.

**The problem: IMDb has no original language.** `title.akas` gives the title
used in each region and language, not the language the film was made in.
Guessing it from `title.akas` was tried first and gave wrong results on
romanised titles: Toshiro Mifune was credited with films in German, Dutch
and Swedish, because Sweden kept the title *Rashomon* while Japan uses the
title in kanji.

**Method.**

1. Successful feature films: rating >= 8.0 with >= 1,000 votes.
2. Their original language is downloaded from **Wikidata**, property
   "original language of film or TV show" (P364), using the IMDb id stored
   in Wikidata (P345). Wikidata is an open (CC0), human-edited database;
   the script `scripts/fetch_languages.py` sends one request per second and
   keeps the Wikidata URL of every film.
3. When Wikidata lists several original languages, the first one is kept
   (statements with the "preferred" rank first).
4. Varieties of one language, which their speakers understand without
   difficulty when spoken, are merged (`LANGUAGE_GROUPS`): Serbian, Croatian
   and Bosnian into Serbo-Croatian, Hindi and Urdu into Hindustani, and
   regional labels (American English, Quebec French...) into their language.
   Cantonese and Mandarin are kept apart: they are not mutually
   intelligible.
5. Actors and actresses (`title.principals`) of these films; a person needs
   successful films in at least two languages.

**Known limits.**

- Films missing from Wikidata or without a language there are left out
  (coverage below).
- The order of the languages on Wikidata follows the order of edition: for
  some multilingual films the first language is not the main one (*The Deer
  Hunter* is listed Russian, English, French).

In [14]:
languages = pd.read_json(ENRICHMENT_DIR / "raw" / "languages.jsonl", lines=True)
print(languages["status"].value_counts().to_string())
print("Films with several languages on Wikidata:",
      int((languages["languages"].str.len() > 1).sum()))

status
ok             1532
no_item         175
no_language     158
Films with several languages on Wikidata: 176


In [15]:
pd.DataFrame(sorted(ex.LANGUAGE_GROUPS.items()), columns=["Wikidata label", "merged into"])

,Wikidata label,merged into
0,American English,English
1,Bosnian,Serbo-Croatian
2,Brazilian Portuguese,Portuguese
3,British English,English
4,Croatian,Serbo-Croatian
5,Egyptian Arabic,Arabic
6,Hindi,Hindustani
7,Mandarin,Mandarin Chinese
8,Modern Greek,Greek
9,Montenegrin,Serbo-Croatian


In [16]:
actors = result("q6_multilingual_actors")
print(f"{actors.count():,} people with >= 2 languages, "
      f"{actors.where('languages >= 3').count():,} with >= 3 languages")
actors.groupBy("languages").count().orderBy("languages").toPandas()

313 people with >= 2 languages, 40 with >= 3 languages


,languages,count
0,2,273
1,3,35
2,4,4
3,5,1


In [17]:
# Actors with successful films in at least three languages.
(actors.where("languages >= 3")
 .select("primaryName", "languages", F.array_join("language_list", ", ").alias("language_list"),
         "successful_films")
 .toPandas())

,primaryName,languages,language_list,successful_films
0,Amrish Puri,5,"English, Hindustani, Malayalam, Tamil, Telugu",7
1,Jayaram,4,"Kannada, Malayalam, Tamil, Telugu",7
2,Ashish Vidyarthi,4,"Hindustani, Malayalam, Tamil, Telugu",6
3,Mukesh Rishi,4,"Hindustani, Marathi, Tamil, Telugu",4
4,Silk Smitha,4,"Hindustani, Malayalam, Tamil, Telugu",4
5,Nassar,3,"Hindustani, Tamil, Telugu",25
6,Kamal Haasan,3,"Hindustani, Tamil, Telugu",23
7,Prakash Raj,3,"Kannada, Tamil, Telugu",19
8,Mammootty,3,"English, Malayalam, Tamil",15
9,Achyuth Kumar,3,"Kannada, Tamil, Telugu",12


In [18]:
# Most frequent languages among the successful films of these actors.
(actors.select(F.explode("language_list").alias("language"))
 .groupBy("language").count().orderBy(F.desc("count")).limit(15).toPandas())

,language,count
0,Tamil,156
1,Hindustani,104
2,Telugu,90
3,Malayalam,87
4,English,43
5,Kannada,27
6,Marathi,20
7,Bangla,19
8,Serbo-Croatian,18
9,French,16


## Q7 – Long careers that peaked early

> Find all actors/actresses who have been in movies for 20+ years and at
> least 15 or more movies whose popularity peaked early in their career.

**Method.**

- **Career**: measured on all the person's released feature films as an
  actor or actress (any number of votes): first and last year at least 20
  years apart, at least 15 films. Films announced for a future year are
  left out.
- **Popularity**: the rating, as in the example of the question ("highest
  rated movies early in their career"), compared only on films with at
  least 1,000 votes.
- **Peaked early**: the best-rated film is in the first half of the career
  **and** the average rating of the first half is higher than the second
  half's.
- To compare the two halves on enough films, the person needs at least 5
  rated films, with at least 2 in each half.

**Why.** Measuring the career on all films follows the question ("been in
movies for 20+ years and at least 15 movies"); using only reliable ratings
avoids comparing ratings given by a few people. Without a minimum number of
rated films, many careers were judged on one film per half.

Alternatives considered: the number of votes as popularity (favours recent
films, so peaks are rarely early), the first third of the career instead of
the first half, or the best film alone.

In [19]:
early = result("q7_early_peak_actors")
print(f"{early.count():,} people")
early.orderBy(F.desc("rating_drop")).limit(20).toPandas()

4,665 people


,nconst,primaryName,first_year,last_year,career_years,movies,rated_movies,best_title,best_year,best_rating,early_avg_rating,late_avg_rating,rating_drop
0,nm0230507,Tatyana Dogileva,1981,2026,45,58,5,A Railway Station for Two,1983,7.8,7.57,3.65,3.92
1,nm0930695,Grant Williams,1956,1976,20,16,8,The Incredible Shrinking Man,1957,7.6,6.52,2.90,3.62
2,nm0179819,Wendell Corey,1947,1968,21,39,22,Rear Window,1954,8.4,7.01,3.65,3.36
3,nm0725338,Ralf Richter,1984,2024,40,30,9,Bang Boom Bang - Ein todsicheres Ding,1999,7.5,6.70,3.45,3.25
4,nm0023868,Zeenat Aman,1970,2025,55,66,10,Don,1978,7.7,6.99,3.80,3.19
5,nm0619870,Reza Naji,1997,2025,28,32,5,Children of Heaven,1997,8.2,7.80,4.70,3.10
6,nm0182661,Nicolas Coster,1964,2020,56,23,11,Reds,1981,7.3,6.42,3.40,3.02
7,nm0747500,Irina Rozanova,1986,2026,40,68,5,The Rifleman of the Voroshilov Regiment,1999,7.4,7.20,4.20,3.00
8,nm1431568,Sphadikam George,1993,2026,33,58,9,Spadikam,1995,8.6,7.49,4.50,2.99
9,nm0929679,Toyah Willcox,1979,2024,45,19,5,Quadrophenia,1979,7.2,6.75,3.77,2.98


In [20]:
# Best known careers in the result (by number of rated films).
early.orderBy(F.desc("rated_movies")).limit(15).toPandas()

,nconst,primaryName,first_year,last_year,career_years,movies,rated_movies,best_title,best_year,best_rating,early_avg_rating,late_avg_rating,rating_drop
0,nm0000821,Amitabh Bachchan,1969,2025,56,190,121,Chupke Chupke,1975,8.3,7.05,6.07,0.99
1,nm0103977,Brahmanandam,1987,2026,39,639,114,Aha Naa Pellanta,1987,8.9,7.37,6.27,1.10
2,nm0080238,Tanikella Bharani,1987,2026,39,340,110,Nuvvu Naaku Nachchav,2001,8.8,7.34,6.28,1.06
3,nm0000168,Samuel L. Jackson,1987,2026,39,112,107,Pulp Fiction,1994,8.8,6.46,6.24,0.22
4,nm0000367,Gérard Depardieu,1971,2026,55,179,102,Jean de Florette,1986,8.1,6.70,6.02,0.68
5,nm0482320,Mohanlal,1981,2026,45,277,101,Kireedam,1989,8.8,8.08,6.18,1.90
6,nm0000246,Bruce Willis,1987,2023,36,101,101,Pulp Fiction,1994,8.8,6.48,4.93,1.55
7,nm0000134,Robert De Niro,1968,2026,58,101,99,The Godfather Part II,1974,9.0,6.91,6.16,0.75
8,nm0000353,Willem Dafoe,1981,2026,45,118,97,Finding Nemo,2003,8.2,6.62,6.36,0.25
9,nm1388202,Siddique,1985,2026,41,252,90,Sandesham,1991,9.0,7.82,6.27,1.55


## Q8 – Multi-talented people

> Identify film professionals who can be considered "multi-talented" (e.g.
> some combo of writer/director/actor). How do these individuals perform when
> performing multiple roles on the same project?

**Method.**

- Three role families: **acting** (`actor`, `actress`), **directing** and
  **writing**, taken from `title.principals` and `title.crew` (which lists
  all directors and writers), on feature films.
- A person is multi-talented when they hold two different role families on
  two **different** films: holding several roles on one single film is not
  enough. This is the same as having at least two role families and at
  least two films.
- Performance (films with >= 1,000 votes):
  1. average rating per combination of roles held on the same film;
  2. for each multi-talented person who has both kinds of films, the
     average rating of their multi-role films against their single-role
     films.

**Why.** Directing and writing count as two talents: they are different
jobs, even if many directors write their own films. Producer and composer
were left out: the question gives writer, director and actor as examples,
and "producer" is often an administrative credit.

In [21]:
people = result("q8_multi_talented_people")
people.groupBy("roles").count().orderBy(F.desc("count")).toPandas()

,roles,count
0,directing+writing,54032
1,acting+directing+writing,32358
2,acting+writing,21705
3,acting+directing,7855


In [22]:
result("q8_role_combination_performance").toPandas()

,combo,person_movies,people,avg_rating,median_rating,avg_votes
0,acting,76827,15570,6.19,6.3,40526.0
1,writing,26617,11328,6.24,6.4,41224.0
2,directing+writing,25796,12183,6.28,6.5,24607.0
3,directing,17015,6817,6.24,6.4,31568.0
4,acting+writing,3320,2285,5.95,6.1,25501.0
5,acting+directing+writing,2444,1505,6.02,6.2,17049.0
6,acting+directing,693,514,6.03,6.3,27196.0


In [23]:
comparison = result("q8_multi_vs_single_role")
comparison.agg(
    F.count(F.lit(1)).alias("people"),
    F.round(F.avg("multi_role_avg_rating"), 2).alias("avg_multi_role"),
    F.round(F.avg("single_role_avg_rating"), 2).alias("avg_single_role"),
    F.round(F.avg("difference"), 2).alias("avg_difference"),
    F.round(100 * F.avg((F.col("difference") > 0).cast("int")), 1).alias("pct_better_multi_role"),
).toPandas()

,people,avg_multi_role,avg_single_role,avg_difference,pct_better_multi_role
0,8694,6.09,6.09,0.0,50.9


In [24]:
# People with at least 5 films of each kind: largest gains and losses.
regular = comparison.where("multi_role_movies >= 5 and single_role_movies >= 5")
pd.concat([
    regular.orderBy(F.desc("difference")).limit(10).toPandas(),
    regular.orderBy("difference").limit(10).toPandas(),
], keys=["best when combining", "worst when combining"])

nconst      primaryName  multi_role_movies  single_role_movies  multi_role_avg_rating  single_role_avg_rating  difference
best when combining  0  nm1962192          Upendra                  7                   5                   8.17                    5.98        2.19
                     1  nm0001385        Eric Idle                  8                  14                   7.39                    5.58        1.81
                     2  nm0001466     Sergio Leone                  7                   6                   7.86                    6.18        1.67
                     3  nm0000092      John Cleese                  8                  46                   7.55                    5.99        1.56
                     4  nm0110292    Fausto Brizzi                  7                  11                   5.77                    4.25        1.52
                     5  nm0386246  Rajkumar Hirani                  6                   5                   7.80                    6.30        1.50
                     6  nm0424103      Karan Johar                  6                  10                   6.48                    5.18        1.30
                     7  nm0482309              Lal                  6                  45                   7.83                    6.61        1.22
                     8  nm0000154       Mel Gibson                  5                  57                   7.42                    6.22        1.20
                     9  nm0000255      Ben Affleck                  8                  44                   7.25                    6.13        1.12
worst when combining 0  nm0518579      Ulli Lommel                  5                   8                   3.68                    6.61       -2.93
                     1  nm0783536    Aaron Seltzer                  8                   7                   3.14                    5.16       -2.02
                     2  nm0294997  Jason Friedberg                  8                   7                   3.14                    5.16       -2.02
                     3  nm1179457     Milap Zaveri                  6                  22                   3.70                    5.03       -1.33
                     4  nm0247642   George Eastman                  6                  20                   4.48                    5.73       -1.24
                     5  nm0222150      Prabhu Deva                  6                  23                   4.73                    5.97       -1.24
                     6  nm1229520      Ben Falcone                  5                   6                   5.20                    6.43       -1.23
                     7  nm0012541   Nikkhil Advani                  6                   5                   5.33                    6.52       -1.19
                     8  nm0700872   Puri Jagannadh                 18                   5                   5.83                    6.96       -1.13
                     9  nm1347153      Tyler Perry                 35                   7                   5.36                    6.43       -1.07

In [25]:
spark.stop()